In [1]:
import pandas as pd
from darts.timeseries import TimeSeries
from darts.utils.timeseries_generation import datetime_attribute_timeseries
from darts.dataprocessing.transformers import Scaler
from darts.models import TFTModel
from darts.dataprocessing.transformers import StaticCovariatesTransformer
import numpy as np
import torch
import matplotlib.pyplot as plt
import joblib
import os

The StatsForecast module could not be imported. To enable support for the AutoARIMA, AutoETS and Croston models, please consider installing it.
The `XGBoost` module could not be imported. To enable XGBoost support in Darts, follow the detailed instructions in the installation guide: https://github.com/unit8co/darts/blob/master/INSTALL.md
The `XGBoost` module could not be imported. To enable XGBoost support in Darts, follow the detailed instructions in the installation guide: https://github.com/unit8co/darts/blob/master/INSTALL.md


### Loss Logger 

In [2]:
from pytorch_lightning.callbacks import Callback

class LossLogger(Callback):
    """
    A PyTorch Lightning callback to record training and validation losses 
    at the end of every epoch for custom plotting or analysis.
    """
    def __init__(self):
        super().__init__()
        self.train_losses = []
        self.val_losses = []

    def on_train_epoch_end(self, trainer, pl_module):
        # Retrieve train_loss from callback_metrics
        train_loss = trainer.callback_metrics.get("train_loss")
        
        if train_loss is not None:
            # detach() ensures we don't keep the computation graph in memory
            # cpu() ensures it works regardless of whether you're on GPU or CPU
            self.train_losses.append(float(train_loss.detach().cpu()))

    def on_validation_epoch_end(self, trainer, pl_module):
        # Retrieve val_loss from callback_metrics
        val_loss = trainer.callback_metrics.get("val_loss")
        
        if val_loss is not None:
            self.val_losses.append(float(val_loss.detach().cpu()))

In [3]:
loss_logger = LossLogger()

### Add encoders

In [4]:
# Function to encode the year as a normalized value
def encode_year(idx):
  return (idx.year - 2000) / 50

def encode_days_in_month(index):
  return index.days_in_month.to_numpy().reshape(-1,1)

# Set up the add_encoders dictionary to specify how different time-related encoders and transformers should be applied
add_encoders = {
    'cyclic': {'past': ['month'], 'future': ['month']},
    'position': {'past': ['relative'], 'future': ['relative']},
    'custom': {
        'past': [encode_year, encode_days_in_month],
        'future': [encode_year, encode_days_in_month]
    },
    'transformer': Scaler()
}

### Read the data

In [5]:
pandas_df = pd.read_csv(r"C:\Users\G0004878\Desktop\TFT_Data\Sep_forecast_Oct_SOQ\Step 1 - Data Preparation\Series_A_data.csv",index_col = ['MONTH_OF_SALE'],parse_dates=True)

In [6]:
pd.set_option('display.max_columns',None)

In [7]:
pandas_df.head()

,PARENT_DEALER_CODE,MODEL_FAMILY,PARENT_DEALER_CODE_MODEL_FAMILY,BRAKE_TYPE,IGNITION_TYPE,WHEEL_TYPE,COLOUR,NET_SALES,DUSSEHRA_(VIJAYADASHAMI)_DAYS,AKSHAYA_TRITIYA_DAYS,BHAI_DOOJ_DAYS,BUDDHA_PURNIMA_DAYS,CHHATH_PUJA_DAYS,DHANTERAS_DAYS,DIWALI_DAYS,EID_UL_FITR_DAYS,GANESH_CHATURTHI_DAYS,GANGA_DUSSEHRA_DAYS,GOVARDHAN_POOJA_DAYS,GURU_PURNIMA_DAYS,HANUMAN_JAYANTI_DAYS,HARTALIK_TEEJ_DAYS,HOLI_DAYS,HOLIKA_DAHAN_DAYS,JAGANNATH_RATHYATRA_DAYS,JANMASHTAMI_DAYS,KARWA_CHAUTH_DAYS,LOHRI_DAYS,MAHA_SHIVARATRI_DAYS,MAKAR_SANKRANTI_PONGAL_DAYS,NAG_PANCHAMI_DAYS,NAVRATRI_DAYS,NEW_YEAR_DAYS,ONAM_DAYS,RAKSHA_BANDHAN_DAYS,REPUBLIC_DAY_DAYS,VASANT_PANCHAMI_DAYS,VISHWAKARMA_PUJA_DAYS,FESTIVE_PHASE_I,FESTIVE_PHASE_II,FESTIVE_PHASE_III,PITRU_PAKSH,PROP_FESTIVE_PHASE_I,PROP_FESTIVE_PHASE_II,PROP_FESTIVE_PHASE_III,PROP_PITRU_PAKSH,DEALER_CITY,X_CITY_CATEGORY,ZONAL_OFFICE_NAME,LAST_YEAR_CONTRIBUTION,MARRIAGE_DAYS
MONTH_OF_SALE,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2023-04-01,12199,DESTINI,12199<>DESTINI<>DRUM<>SELF<>CAST<>BLUE,DRUM,SELF,CAST,BLUE,3.0,0.0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.00000,0.00000,0.0,CHENNAI,URBAN,Zonal Office - South,0.0935,1.0
2023-11-01,12199,DESTINI,12199<>DESTINI<>DRUM<>SELF<>CAST<>BLUE,DRUM,SELF,CAST,BLUE,0.0,0.0,0,1,0,1,1,1,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0.0,8.0,7.0,0.0,0.0,0.26667,0.23333,0.0,CHENNAI,URBAN,Zonal Office - South,0.0566,8.0
2026-06-01,12199,DESTINI,12199<>DESTINI<>DRUM<>SELF<>CAST<>BLUE,DRUM,SELF,CAST,BLUE,3.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.00000,0.00000,0.0,CHENNAI,URBAN,Zonal Office - South,0.0636,11.0
2024-07-01,12199,DESTINI,12199<>DESTINI<>DRUM<>SELF<>CAST<>BLUE,DRUM,SELF,CAST,BLUE,0.0,0.0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.00000,0.00000,0.0,CHENNAI,URBAN,Zonal Office - South,0.0611,5.0
2024-03-01,12199,DESTINI,12199<>DESTINI<>DRUM<>SELF<>CAST<>BLUE,DRUM,SELF,CAST,BLUE,0.0,0.0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.00000,0.00000,0.0,CHENNAI,URBAN,Zonal Office - South,0.0972,3.0


In [8]:
print(f"Minimum date in the data is {pandas_df.index.min()}")

print(f"Maximum date in the data is {pandas_df.index.max()}")

Minimum date in the data is 2023-04-01 00:00:00
Maximum date in the data is 2026-10-01 00:00:00


### Separating the type of variables

In [49]:
#past_covariates : Historical known : Future unknown (Market share)
#future_covariates : Historical known : Future known (Marketing expenditure)
#static cols : Dealer city, Zonal office, 

In [9]:
#Extracting type of columns according to the datatypes
# 1. Targets/Metrics (The numbers we want to predict)
target_cols = pandas_df.select_dtypes(include=['number']).columns.tolist()
target_cols.append('PARENT_DEALER_CODE_MODEL_FAMILY')

# 2. Time Dimension
time_cols = pandas_df.select_dtypes(include=['datetime', 'datetime64']).columns.tolist()

# 3. Static/Categorical Covariates (The identifiers)
# We exclude numbers and dates to find the "ID" strings
static_cols = pandas_df.select_dtypes(exclude=['number', 'datetime', 'datetime64']).columns.tolist()
static_cols.append('PARENT_DEALER_CODE')

print(f"Targets: {target_cols}")
print(f"Time Column: {time_cols}")
print(f"Static Identifiers: {static_cols}")

Targets: ['PARENT_DEALER_CODE', 'NET_SALES', 'DUSSEHRA_(VIJAYADASHAMI)_DAYS', 'AKSHAYA_TRITIYA_DAYS', 'BHAI_DOOJ_DAYS', 'BUDDHA_PURNIMA_DAYS', 'CHHATH_PUJA_DAYS', 'DHANTERAS_DAYS', 'DIWALI_DAYS', 'EID_UL_FITR_DAYS', 'GANESH_CHATURTHI_DAYS', 'GANGA_DUSSEHRA_DAYS', 'GOVARDHAN_POOJA_DAYS', 'GURU_PURNIMA_DAYS', 'HANUMAN_JAYANTI_DAYS', 'HARTALIK_TEEJ_DAYS', 'HOLI_DAYS', 'HOLIKA_DAHAN_DAYS', 'JAGANNATH_RATHYATRA_DAYS', 'JANMASHTAMI_DAYS', 'KARWA_CHAUTH_DAYS', 'LOHRI_DAYS', 'MAHA_SHIVARATRI_DAYS', 'MAKAR_SANKRANTI_PONGAL_DAYS', 'NAG_PANCHAMI_DAYS', 'NAVRATRI_DAYS', 'NEW_YEAR_DAYS', 'ONAM_DAYS', 'RAKSHA_BANDHAN_DAYS', 'REPUBLIC_DAY_DAYS', 'VASANT_PANCHAMI_DAYS', 'VISHWAKARMA_PUJA_DAYS', 'FESTIVE_PHASE_I', 'FESTIVE_PHASE_II', 'FESTIVE_PHASE_III', 'PITRU_PAKSH', 'PROP_FESTIVE_PHASE_I', 'PROP_FESTIVE_PHASE_II', 'PROP_FESTIVE_PHASE_III', 'PROP_PITRU_PAKSH', 'LAST_YEAR_CONTRIBUTION', 'MARRIAGE_DAYS', 'PARENT_DEALER_CODE_MODEL_FAMILY']
Time Column: []
Static Identifiers: ['MODEL_FAMILY', 'PARENT_DEA

In [10]:
#Separating the covariates
target_col = ["NET_SALES"]

#future covariates
future_covariates = [i for i in target_cols if i not in ['NET_SALES','PARENT_DEALER_CODE']]

#actual_static_cols
actual_static_cols = [i for i in static_cols if i!='PARENT_DEALER_CODE_MODEL_FAMILY']

In [11]:
future_covariates

['DUSSEHRA_(VIJAYADASHAMI)_DAYS',
 'AKSHAYA_TRITIYA_DAYS',
 'BHAI_DOOJ_DAYS',
 'BUDDHA_PURNIMA_DAYS',
 'CHHATH_PUJA_DAYS',
 'DHANTERAS_DAYS',
 'DIWALI_DAYS',
 'EID_UL_FITR_DAYS',
 'GANESH_CHATURTHI_DAYS',
 'GANGA_DUSSEHRA_DAYS',
 'GOVARDHAN_POOJA_DAYS',
 'GURU_PURNIMA_DAYS',
 'HANUMAN_JAYANTI_DAYS',
 'HARTALIK_TEEJ_DAYS',
 'HOLI_DAYS',
 'HOLIKA_DAHAN_DAYS',
 'JAGANNATH_RATHYATRA_DAYS',
 'JANMASHTAMI_DAYS',
 'KARWA_CHAUTH_DAYS',
 'LOHRI_DAYS',
 'MAHA_SHIVARATRI_DAYS',
 'MAKAR_SANKRANTI_PONGAL_DAYS',
 'NAG_PANCHAMI_DAYS',
 'NAVRATRI_DAYS',
 'NEW_YEAR_DAYS',
 'ONAM_DAYS',
 'RAKSHA_BANDHAN_DAYS',
 'REPUBLIC_DAY_DAYS',
 'VASANT_PANCHAMI_DAYS',
 'VISHWAKARMA_PUJA_DAYS',
 'FESTIVE_PHASE_I',
 'FESTIVE_PHASE_II',
 'FESTIVE_PHASE_III',
 'PITRU_PAKSH',
 'PROP_FESTIVE_PHASE_I',
 'PROP_FESTIVE_PHASE_II',
 'PROP_FESTIVE_PHASE_III',
 'PROP_PITRU_PAKSH',
 'LAST_YEAR_CONTRIBUTION',
 'MARRIAGE_DAYS',
 'PARENT_DEALER_CODE_MODEL_FAMILY']

In [12]:
actual_static_cols

['MODEL_FAMILY',
 'BRAKE_TYPE',
 'IGNITION_TYPE',
 'WHEEL_TYPE',
 'COLOUR',
 'DEALER_CITY',
 'X_CITY_CATEGORY',
 'ZONAL_OFFICE_NAME',
 'PARENT_DEALER_CODE']

In [13]:
# #since variables like MODEL_FAMILY,BRAKE_VARIANT,IGNITION_TYPE,WHEEL_TYPE,BIKE_COLOUR are mostly same for all the top 10 series, will be removing them from the static covariates'
# static_covariates = [i for i in actual_static_cols if i not in ['MODEL_FAMILY','BRAKE_VARIANT','IGNITION_TYPE','WHEEL_TYPE','BIKE_COLOUR','DEALER_CODE']]
# static_covariates

static_covariates = actual_static_cols.copy()

static_covariates

['MODEL_FAMILY',
 'BRAKE_TYPE',
 'IGNITION_TYPE',
 'WHEEL_TYPE',
 'COLOUR',
 'DEALER_CITY',
 'X_CITY_CATEGORY',
 'ZONAL_OFFICE_NAME',
 'PARENT_DEALER_CODE']

### Preparing data for Darts

In [14]:
#Step 1 - Sorting the dataframe by date
pandas_df=pandas_df.reset_index().sort_values(by=["PARENT_DEALER_CODE_MODEL_FAMILY","MONTH_OF_SALE"]).set_index("MONTH_OF_SALE")

In [15]:
#Step 2 - Separating the static covariates and NET_SALES column
pandas_df_with_target_and_static_covariates = pandas_df.loc[:,['PARENT_DEALER_CODE_MODEL_FAMILY','NET_SALES']+static_covariates]
pandas_df_with_target_and_static_covariates.head()

,PARENT_DEALER_CODE_MODEL_FAMILY,NET_SALES,MODEL_FAMILY,BRAKE_TYPE,IGNITION_TYPE,WHEEL_TYPE,COLOUR,DEALER_CITY,X_CITY_CATEGORY,ZONAL_OFFICE_NAME,PARENT_DEALER_CODE
MONTH_OF_SALE,,,,,,,,,,,
2023-04-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK,4.0,DESTINI,DRUM,SELF,CAST,BLACK,AMRITSAR,URBAN,Zonal Office - North,10001
2023-05-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK,6.0,DESTINI,DRUM,SELF,CAST,BLACK,AMRITSAR,URBAN,Zonal Office - North,10001
2023-06-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK,4.0,DESTINI,DRUM,SELF,CAST,BLACK,AMRITSAR,URBAN,Zonal Office - North,10001
2023-07-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK,5.0,DESTINI,DRUM,SELF,CAST,BLACK,AMRITSAR,URBAN,Zonal Office - North,10001
2023-08-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK,3.0,DESTINI,DRUM,SELF,CAST,BLACK,AMRITSAR,URBAN,Zonal Office - North,10001


In [16]:
#Step 3 - Separating the future covariates
pandas_df_with_future_covariates = pandas_df.loc[:,future_covariates]
pandas_df_with_future_covariates.head()

,DUSSEHRA_(VIJAYADASHAMI)_DAYS,AKSHAYA_TRITIYA_DAYS,BHAI_DOOJ_DAYS,BUDDHA_PURNIMA_DAYS,CHHATH_PUJA_DAYS,DHANTERAS_DAYS,DIWALI_DAYS,EID_UL_FITR_DAYS,GANESH_CHATURTHI_DAYS,GANGA_DUSSEHRA_DAYS,GOVARDHAN_POOJA_DAYS,GURU_PURNIMA_DAYS,HANUMAN_JAYANTI_DAYS,HARTALIK_TEEJ_DAYS,HOLI_DAYS,HOLIKA_DAHAN_DAYS,JAGANNATH_RATHYATRA_DAYS,JANMASHTAMI_DAYS,KARWA_CHAUTH_DAYS,LOHRI_DAYS,MAHA_SHIVARATRI_DAYS,MAKAR_SANKRANTI_PONGAL_DAYS,NAG_PANCHAMI_DAYS,NAVRATRI_DAYS,NEW_YEAR_DAYS,ONAM_DAYS,RAKSHA_BANDHAN_DAYS,REPUBLIC_DAY_DAYS,VASANT_PANCHAMI_DAYS,VISHWAKARMA_PUJA_DAYS,FESTIVE_PHASE_I,FESTIVE_PHASE_II,FESTIVE_PHASE_III,PITRU_PAKSH,PROP_FESTIVE_PHASE_I,PROP_FESTIVE_PHASE_II,PROP_FESTIVE_PHASE_III,PROP_PITRU_PAKSH,LAST_YEAR_CONTRIBUTION,MARRIAGE_DAYS,PARENT_DEALER_CODE_MODEL_FAMILY
MONTH_OF_SALE,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2023-04-01,0.0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0935,1.0,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK
2023-05-01,0.0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.1047,17.0,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK
2023-06-01,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0791,6.0,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK
2023-07-01,0.0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0604,0.0,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK
2023-08-01,0.0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0617,0.0,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK


In [17]:
#Step 4 - Creating the darts timeseries object for target and static covariates
darts_df_with_static_covariates = TimeSeries.from_group_dataframe(df=pandas_df_with_target_and_static_covariates,
                                                                  group_cols=["PARENT_DEALER_CODE_MODEL_FAMILY"],
                                                                  static_cols=static_covariates,value_cols=["NET_SALES"],freq='MS')


In [18]:
#Step 5 - Creating the darts timeseries object with future covariates

#Removing PARENT_DEALER_CODE_MODEL_FAMILY from future_covariates
try:
    future_covariates.remove('PARENT_DEALER_CODE_MODEL_FAMILY')
except:
    pass

darts_df_with_future_covariates = TimeSeries.from_group_dataframe(df = pandas_df_with_future_covariates,
                                    group_cols="PARENT_DEALER_CODE_MODEL_FAMILY",
                                    freq = 'MS',
                                    value_cols = future_covariates
                                    )

### Train/Test split

In [19]:
#Step 6 - Creating train, test, and validation split
#Train set - Apr'23 to Dec'25 
#Val set - Jan'26 to Mar'26 


train_list = []
val_list = []

for ts in darts_df_with_static_covariates:
    train = ts.slice(pd.Timestamp('2023-04-01'), pd.Timestamp('2025-12-01'))
    val = ts.slice(pd.Timestamp('2024-09-01'), pd.Timestamp('2026-08-01'))
    
    train_list.append(train)
    val_list.append(val)

train_future_covariates_list = []
validation_future_covariates_list = []

for ts in darts_df_with_future_covariates:
    train = ts.slice(pd.Timestamp('2023-04-01'), pd.Timestamp('2025-12-01'))
    val = ts.slice(pd.Timestamp('2024-09-01'), pd.Timestamp('2026-08-01'))
    train_future_covariates_list.append(train)
    validation_future_covariates_list.append(val)

In [20]:
target_scaler = Scaler(n_jobs=-1)
future_covariates_scaler = Scaler(n_jobs=-1)

transformer = StaticCovariatesTransformer(n_jobs=-1)

#Scale the target training data
scaled_target_series = target_scaler.fit_transform(train_list)

scaled_target_series_with_static_covariates_training = transformer.fit_transform(scaled_target_series)



# #Scale the static covariates in training data
# scaled_static_covariates_training = transformer.fit_transform(train_list)

# #Scale the future covariates in training data
# # scaled_future_covariates = future_covariates_scaler.fit_transform(darts_df_with_future_covariates)

scaled_future_covariates_training = future_covariates_scaler.fit_transform(train_future_covariates_list)
scaled_future_covariates_validation = future_covariates_scaler.transform(validation_future_covariates_list)


# #Scale the target validation data
scaled_target_series_validation = target_scaler.transform(val_list)
scaled_target_series_with_static_covariates_validation = transformer.transform(scaled_target_series_validation)

# #Scale the static covariates in validation data
# scaled_static_covariates_validation = transformer.transform(val_list)


In [21]:
os.makedirs("scaled_objects",exist_ok=True)
output_dir_scaler = os.path.join(os.getcwd(),"scaled_objects")
joblib.dump(target_scaler, os.path.join(output_dir_scaler, 'target_scaler.pkl'))
joblib.dump(future_covariates_scaler, os.path.join(output_dir_scaler, 'future_covariates_scaler.pkl'))
joblib.dump(transformer, os.path.join(output_dir_scaler, 'static_transformer.pkl'))

['c:\\Users\\G0004878\\Desktop\\TFT_Data\\Sep_forecast_Oct_SOQ\\Step 2 - Modelling\\scaled_objects\\static_transformer.pkl']

### Casting data types

In [22]:
def force_float32_target(ts):
    """Targets: keep static covariates (already numeric via
    StaticCovariatesTransformer), cast both values and statics to float32."""
    ts = ts.astype(np.float32)
    if ts.has_static_covariates:
        ts = ts.with_static_covariates(ts.static_covariates.astype(np.float32))
    return ts

def force_float32_cov(ts):
    """Future covariates: from_group_dataframe attaches the group key as a
    STRING static covariate. Darts reads statics from the target series only,
    so drop them here, then cast values."""
    return ts.with_static_covariates(None).astype(np.float32)


scaled_target_series_with_static_covariates_training = [
    force_float32_target(ts) for ts in scaled_target_series_with_static_covariates_training
]
scaled_target_series_with_static_covariates_validation = [
    force_float32_target(ts) for ts in scaled_target_series_with_static_covariates_validation
]
scaled_future_covariates_training = [
    force_float32_cov(ts) for ts in scaled_future_covariates_training
]
scaled_future_covariates_validation = [
    force_float32_cov(ts) for ts in scaled_future_covariates_validation
]

for name, lst in [("tgt train", scaled_target_series_with_static_covariates_training),
                  ("tgt val",   scaled_target_series_with_static_covariates_validation),
                  ("cov train", scaled_future_covariates_training),
                  ("cov val",   scaled_future_covariates_validation)]:
    ts = lst[0]
    sc = ts.static_covariates.dtypes.unique().tolist() if ts.has_static_covariates else "none"
    print(f"{name:10s} n={len(lst):6,}  values={ts.dtype}  statics={sc}")


tgt train  n=40,160  values=float32  statics=[dtype('float32')]
tgt val    n=40,160  values=float32  statics=[dtype('float32')]
cov train  n=40,160  values=float32  statics=none
cov val    n=40,160  values=float32  statics=none


In [23]:
from datetime import datetime

from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from darts.models import TFTModel

In [24]:
from datetime import datetime
from pytorch_lightning.callbacks import ModelCheckpoint

now = datetime.now().strftime("%Y-%m-%d_%H_%M_%S")


WORK_DIR = os.getcwd()
MODEL_NAME = f"tft_sep_forecast_oct_soq_using_data_till_August_{now}"

MODEL_DIR = os.path.join(WORK_DIR, MODEL_NAME)
CHECKPOINT_DIR = os.path.join(MODEL_DIR, "checkpoints")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("MODEL_DIR:", MODEL_DIR)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)

MODEL_DIR: c:\Users\G0004878\Desktop\TFT_Data\Sep_forecast_Oct_SOQ\Step 2 - Modelling\tft_sep_forecast_oct_soq_using_data_till_August_2026-09-01_14_44_45
CHECKPOINT_DIR: c:\Users\G0004878\Desktop\TFT_Data\Sep_forecast_Oct_SOQ\Step 2 - Modelling\tft_sep_forecast_oct_soq_using_data_till_August_2026-09-01_14_44_45\checkpoints


In [25]:
class DateStampedCheckpoint(ModelCheckpoint):
    @property
    def state_key(self) -> str:
        return f"DateStampedCheckpoint_{self.monitor}_{self.dirpath}"

In [26]:
checkpoint_callback = DateStampedCheckpoint(
    dirpath=CHECKPOINT_DIR,
    filename="best_model",          # Completely static name: saves as best_model.ckpt
    monitor="val_loss",
    mode="min",
    save_top_k=1,
    save_last=True,                 # Also retains 'last.ckpt' as a fallback backup
    verbose=True
)

In [27]:
early_stop_callback = EarlyStopping(
    monitor="val_loss",
    patience=10,
    mode="min",
    verbose=True
)

In [28]:
loss_logger = LossLogger()

if torch.cuda.is_bf16_supported():
    print("Awesome! bf16 is supported. Using bf16-mixed.")
    precision_setting = "bf16-mixed"
else:
    print("Warning: bf16 is not supported on this GPU. Falling back to 16-mixed.")
    precision_setting = "16-mixed"

model = TFTModel(
    input_chunk_length=16,
    output_chunk_length=2,
    batch_size=256,
    dropout=0.1,
    likelihood=None,
    loss_fn=torch.nn.HuberLoss(delta=1.0),
    n_epochs=100,
    random_state=42,
    add_encoders=add_encoders,
    model_name=MODEL_NAME,
    work_dir=WORK_DIR,
    use_reversible_instance_norm=True,
    
    # CRITICAL CHANGE: Tell Darts to handle its native model manifest building
    save_checkpoints=True,          
    force_reset=True,
    
    pl_trainer_kwargs={
        "callbacks": [
            loss_logger,
            early_stop_callback
        ],
        "enable_checkpointing": True,
        "gradient_clip_val": 0.1,
        "accelerator": "gpu", 
        "devices": [0],
        "precision": precision_setting
    }
)

print("\nRunning LR Finder...")
lr_finder = model.lr_find(
    series=scaled_target_series_with_static_covariates_training,
    future_covariates=scaled_future_covariates_training
)

suggested_lr = lr_finder.suggestion()
print("Suggested Learning Rate:", suggested_lr)
model.lr = suggested_lr

Awesome! bf16 is supported. Using bf16-mixed.

Running LR Finder...


Detected user-defined float16-like precision. For mixed precision training, recommended options are 'bf16-mixed' and '16-mixed'.
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA RTX 2000 Ada Generation') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Finding best initial lr:   0%|          | 0/100 [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=100` reached.
Restoring states from the checkpoint path at c:\Users\G0004878\Desktop\TFT_Data\Sep_forecast_Oct_SOQ\Step 2 - Modelling\.lr_find_39cd5083-ffd3-49a8-a514-eb781b249ed1.ckpt
Restored all states from the checkpoint at c:\Users\G0004878\Desktop\TFT_Data\Sep_forecast_Oct_SOQ\Step 2 - Modelling\.lr_find_39cd5083-ffd3-49a8-a514-eb781b249ed1.ckpt


Suggested Learning Rate: 0.003981071705534969


In [29]:
print(CHECKPOINT_DIR)

c:\Users\G0004878\Desktop\TFT_Data\Sep_forecast_Oct_SOQ\Step 2 - Modelling\tft_sep_forecast_oct_soq_using_data_till_August_2026-09-01_14_44_45\checkpoints


In [30]:
v = np.concatenate([ts.values().ravel()
                    for ts in scaled_target_series_with_static_covariates_training[:2000]])
print(f"scaled: mean {v.mean():.3f}  median {np.median(v):.3f}")
print(f"zero months: {(v == 0).mean()*100:.1f}%")
print(f"share of values above delta=0.1: {(v > 0.1).mean()*100:.1f}%")

scaled: mean 0.199  median 0.116
zero months: 34.2%
share of values above delta=0.1: 52.3%


In [ ]:
print("\nStarting Training with Validation...")
model.fit(
    series=scaled_target_series_with_static_covariates_training,
    future_covariates=scaled_future_covariates_training,
    val_series=scaled_target_series_with_static_covariates_validation,
    val_future_covariates=scaled_future_covariates_validation,
    dataloader_kwargs={
        "num_workers": 4,         # Parallellize data processing on GPU
        "pin_memory": True        # Fast page-locked VRAM transfers
    },
    verbose=True
)

print(f"\n✅ Training Complete. Best model saved at:\n--> {os.path.join(CHECKPOINT_DIR, 'best_model.ckpt')}")


Starting Training with Validation...


Detected user-defined float16-like precision. For mixed precision training, recommended options are 'bf16-mixed' and '16-mixed'.
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

   | Name                              | Type                             | Params | Mode 
------------------------------------------------------------------------------------------------
0  | criterion                         | HuberLoss                        | 0      | train
1  | train_criterion                   | HuberLoss                        | 0      | train
2  | val_criterion                     | HuberLoss                        | 0      | train
3  | train_metrics                     | MetricCollection                 | 0      | train
4  | val_metrics                       | MetricCollection                 | 0      | train
5  | rin              

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

c:\Users\G0004878\Desktop\Virtual_environments\darts_gpu\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:420: Consider setting `persistent_workers=True` in 'val_dataloader' to speed up the dataloader worker initialization.
c:\Users\G0004878\Desktop\Virtual_environments\darts_gpu\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:420: Consider setting `persistent_workers=True` in 'train_dataloader' to speed up the dataloader worker initialization.


Training: |          | 0/? [00:00<?, ?it/s]

### Prediction

In [64]:
from datetime import datetime
from pytorch_lightning.callbacks import ModelCheckpoint


WORK_DIR = os.getcwd()
MODEL_NAME = 'tft_august_sep_forecast_using_data_till_July_2026-08-06_13_55_24'
MODEL_DIR = os.path.join(WORK_DIR, MODEL_NAME)
CHECKPOINT_DIR = os.path.join(MODEL_DIR, "checkpoints")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("MODEL_DIR:", MODEL_DIR)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)

MODEL_DIR: c:\Users\G0004878\Desktop\TFT_Data\Aug_forecast_Sep_SOQ\Step 2 - Modelling\tft_august_sep_forecast_using_data_till_July_2026-08-06_13_55_24
CHECKPOINT_DIR: c:\Users\G0004878\Desktop\TFT_Data\Aug_forecast_Sep_SOQ\Step 2 - Modelling\tft_august_sep_forecast_using_data_till_July_2026-08-06_13_55_24\checkpoints


In [65]:
MODEL_NAME

'tft_august_sep_forecast_using_data_till_July_2026-08-06_13_55_24'

In [66]:
from darts.models import TFTModel

# Direct native restoration call
loaded_model = TFTModel.load_from_checkpoint(model_name=MODEL_NAME,best=True,map_location="cpu")
print("Model loaded successfully with all dimensions intact!")

Model loaded successfully with all dimensions intact!


### Creating the prediction data

In [67]:
prediction_df = pandas_df.loc[((pandas_df.index>='2025-04-01') & (pandas_df.index<='2026-09-01'))]
prediction_df.shape

(709524, 51)

In [68]:
target_plus_static_cols = target_col + static_cols
target_plus_static_cols

['NET_SALES',
 'MODEL_FAMILY',
 'PARENT_DEALER_CODE_MODEL_FAMILY',
 'BRAKE_TYPE',
 'IGNITION_TYPE',
 'WHEEL_TYPE',
 'COLOUR',
 'DEALER_CITY',
 'X_CITY_CATEGORY',
 'ZONAL_OFFICE_NAME',
 'PARENT_DEALER_CODE']

In [69]:
static_plus_future_cov = static_cols + future_covariates
static_plus_future_cov

['MODEL_FAMILY',
 'PARENT_DEALER_CODE_MODEL_FAMILY',
 'BRAKE_TYPE',
 'IGNITION_TYPE',
 'WHEEL_TYPE',
 'COLOUR',
 'DEALER_CITY',
 'X_CITY_CATEGORY',
 'ZONAL_OFFICE_NAME',
 'PARENT_DEALER_CODE',
 'DUSSEHRA_(VIJAYADASHAMI)_DAYS',
 'AKSHAYA_TRITIYA_DAYS',
 'BHAI_DOOJ_DAYS',
 'BUDDHA_PURNIMA_DAYS',
 'CHHATH_PUJA_DAYS',
 'DHANTERAS_DAYS',
 'DIWALI_DAYS',
 'EID_UL_FITR_DAYS',
 'GANESH_CHATURTHI_DAYS',
 'GANGA_DUSSEHRA_DAYS',
 'GOVARDHAN_POOJA_DAYS',
 'GURU_PURNIMA_DAYS',
 'HANUMAN_JAYANTI_DAYS',
 'HARTALIK_TEEJ_DAYS',
 'HOLI_DAYS',
 'HOLIKA_DAHAN_DAYS',
 'JAGANNATH_RATHYATRA_DAYS',
 'JANMASHTAMI_DAYS',
 'KARWA_CHAUTH_DAYS',
 'LOHRI_DAYS',
 'MAHA_SHIVARATRI_DAYS',
 'MAKAR_SANKRANTI_PONGAL_DAYS',
 'NAG_PANCHAMI_DAYS',
 'NAVRATRI_DAYS',
 'NEW_YEAR_DAYS',
 'ONAM_DAYS',
 'RAKSHA_BANDHAN_DAYS',
 'REPUBLIC_DAY_DAYS',
 'VASANT_PANCHAMI_DAYS',
 'VISHWAKARMA_PUJA_DAYS',
 'FESTIVE_PHASE_I',
 'FESTIVE_PHASE_II',
 'FESTIVE_PHASE_III',
 'PITRU_PAKSH',
 'PROP_FESTIVE_PHASE_I',
 'PROP_FESTIVE_PHASE_II',
 'PR

In [70]:
static_covariates

['MODEL_FAMILY',
 'BRAKE_TYPE',
 'IGNITION_TYPE',
 'WHEEL_TYPE',
 'COLOUR',
 'DEALER_CITY',
 'X_CITY_CATEGORY',
 'ZONAL_OFFICE_NAME',
 'PARENT_DEALER_CODE']

In [71]:
future_covariates

['DUSSEHRA_(VIJAYADASHAMI)_DAYS',
 'AKSHAYA_TRITIYA_DAYS',
 'BHAI_DOOJ_DAYS',
 'BUDDHA_PURNIMA_DAYS',
 'CHHATH_PUJA_DAYS',
 'DHANTERAS_DAYS',
 'DIWALI_DAYS',
 'EID_UL_FITR_DAYS',
 'GANESH_CHATURTHI_DAYS',
 'GANGA_DUSSEHRA_DAYS',
 'GOVARDHAN_POOJA_DAYS',
 'GURU_PURNIMA_DAYS',
 'HANUMAN_JAYANTI_DAYS',
 'HARTALIK_TEEJ_DAYS',
 'HOLI_DAYS',
 'HOLIKA_DAHAN_DAYS',
 'JAGANNATH_RATHYATRA_DAYS',
 'JANMASHTAMI_DAYS',
 'KARWA_CHAUTH_DAYS',
 'LOHRI_DAYS',
 'MAHA_SHIVARATRI_DAYS',
 'MAKAR_SANKRANTI_PONGAL_DAYS',
 'NAG_PANCHAMI_DAYS',
 'NAVRATRI_DAYS',
 'NEW_YEAR_DAYS',
 'ONAM_DAYS',
 'RAKSHA_BANDHAN_DAYS',
 'REPUBLIC_DAY_DAYS',
 'VASANT_PANCHAMI_DAYS',
 'VISHWAKARMA_PUJA_DAYS',
 'FESTIVE_PHASE_I',
 'FESTIVE_PHASE_II',
 'FESTIVE_PHASE_III',
 'PITRU_PAKSH',
 'PROP_FESTIVE_PHASE_I',
 'PROP_FESTIVE_PHASE_II',
 'PROP_FESTIVE_PHASE_III',
 'PROP_PITRU_PAKSH',
 'LAST_YEAR_CONTRIBUTION',
 'MARRIAGE_DAYS']

In [72]:
#Step 1 - Preparing the lookback data for the model
lookback_data_pandas_df = prediction_df.loc[((prediction_df.index>='2025-04-01')&(prediction_df.index<='2026-07-01')),target_plus_static_cols]

#Step 2 - Preparing the lookahead data for the model
lookahead_data_pandas_df = prediction_df.loc[((prediction_df.index>='2025-04-01')&(prediction_df.index<='2026-09-01')),static_plus_future_cov]

#Step 3 - Creating the darts time-series object from lookback data for the model
lookback_data_darts_df = TimeSeries.from_group_dataframe(df=lookback_data_pandas_df,
                                                                  group_cols=["PARENT_DEALER_CODE_MODEL_FAMILY"],
                                                                  static_cols=static_covariates,value_cols=["NET_SALES"],freq='MS')

#Step 4 - Creating the darts time-series object from lookahead data for the model 
lookahead_data_darts_df = TimeSeries.from_group_dataframe(
    df=lookahead_data_pandas_df,
    group_cols="PARENT_DEALER_CODE_MODEL_FAMILY",
    static_cols=static_covariates, 
    value_cols=future_covariates,
    freq='MS'
)

In [73]:
scaled_temporal = future_covariates_scaler.transform(lookahead_data_darts_df)

final_scaled_lookahead_data = transformer.transform(scaled_temporal)

In [74]:
target_scaled_data = target_scaler.transform(lookback_data_darts_df)

final_scaled_lookback_data = transformer.transform(target_scaled_data)

In [76]:
# same casts as training
final_scaled_lookback_data  = [force_float32_target(ts) for ts in final_scaled_lookback_data]
final_scaled_lookahead_data = [force_float32_cov(ts)    for ts in final_scaled_lookahead_data]

for name, lst in [("lookback", final_scaled_lookback_data),
                  ("lookahead", final_scaled_lookahead_data)]:
    ts = lst[0]
    sc = ts.static_covariates.dtypes.unique().tolist() if ts.has_static_covariates else "none"
    print(f"{name:10s} n={len(lst):6,}  values={ts.dtype}  statics={sc}")

lookback   n=39,418  values=float32  statics=[dtype('float32')]
lookahead  n=39,418  values=float32  statics=none


In [77]:
forecast_series = loaded_model.predict(
    n=2, 
    series=final_scaled_lookback_data, 
    future_covariates=final_scaled_lookahead_data
)

print("Forecast generated successfully!")

Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
c:\Users\G0004878\Desktop\Virtual_environments\darts_gpu\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.


Predicting: |          | 0/? [00:00<?, ?it/s]

Forecast generated successfully!


In [78]:
actual_pred_series = target_scaler.inverse_transform(forecast_series)

In [79]:
# Build output DataFrame for Apr'26 to Jun'26 predictions
records = []

for forecast, source_series in zip(actual_pred_series, lookahead_data_darts_df):
    # val_list retains original static covariates — use it as the label source
    series_name = source_series.static_covariates['PARENT_DEALER_CODE_MODEL_FAMILY'].values[0]

    months          = forecast.time_index
    forecast_values = forecast.values().flatten()

    for month, pred in zip(months, forecast_values):
        records.append({
            'MONTH_OF_SALE'                   : month,
            'PARENT_DEALER_CODE_MODEL_FAMILY'  : series_name,
            'PREDICTED_SALES'                  : round(float(pred), 2)
        })

df_final_output = pd.DataFrame(records)
df_final_output['MONTH_OF_SALE'] = pd.to_datetime(df_final_output['MONTH_OF_SALE']).dt.strftime('%Y-%m-%d')
df_final_output = df_final_output.sort_values(['PARENT_DEALER_CODE_MODEL_FAMILY', 'MONTH_OF_SALE']).reset_index(drop=True)

print(f'Output shape : {df_final_output.shape}')
print(f'Months       : {df_final_output["MONTH_OF_SALE"].unique()}')
print(f'Series count : {df_final_output["PARENT_DEALER_CODE_MODEL_FAMILY"].nunique()}')
df_final_output.head(10)

Output shape : (78836, 3)
Months       : ['2026-08-01' '2026-09-01']
Series count : 39418


,MONTH_OF_SALE,PARENT_DEALER_CODE_MODEL_FAMILY,PREDICTED_SALES
0,2026-08-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK,2.72
1,2026-09-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK,2.00
2,2026-08-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>WHITE,6.99
3,2026-09-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>WHITE,5.68
4,2026-08-01,10001<>DESTINI<>DRUM<>SELF<>SHEET METAL<>BLACK,11.72
5,2026-09-01,10001<>DESTINI<>DRUM<>SELF<>SHEET METAL<>BLACK,8.48
6,2026-08-01,10001<>DESTINI<>DRUM<>SELF<>SHEET METAL<>BLUE,0.73
7,2026-09-01,10001<>DESTINI<>DRUM<>SELF<>SHEET METAL<>BLUE,0.42
8,2026-08-01,10001<>DESTINI<>DRUM<>SELF<>SHEET METAL<>RED,0.32
9,2026-09-01,10001<>DESTINI<>DRUM<>SELF<>SHEET METAL<>RED,0.23


In [80]:
df_final_output["MONTH_OF_SALE"] = pd.to_datetime(df_final_output["MONTH_OF_SALE"])

In [81]:
df_final_output["MONTH_NAME"] = df_final_output["MONTH_OF_SALE"].dt.strftime('%B')

In [82]:
df_final_output.groupby("MONTH_NAME",as_index=False).agg(TOTAL_MONTHLY_SALES=("PREDICTED_SALES",'sum'))

,MONTH_NAME,TOTAL_MONTHLY_SALES
0,August,327822.36
1,September,297320.66


In [67]:
df_final_output.groupby("MONTH_NAME",as_index=False).agg(TOTAL_MONTHLY_SALES=("PREDICTED_SALES",'sum'))

,MONTH_NAME,TOTAL_MONTHLY_SALES
0,August,311148.39
1,September,260361.83


In [68]:
df_final_output.to_csv(r"Final_output_Series_A_TFT_July_August.csv",index=False)

In [69]:
agg_output = df_final_output.groupby("MONTH_NAME",as_index=False).agg(TOTAL_MONTHLY_SALES=("PREDICTED_SALES",'sum'))

agg_output.to_csv(r"Aggregated_output.csv",index=False)

In [70]:
df_final_output.groupby("MONTH_NAME",as_index=False).agg(TOTAL_MONTHLY_SALES=("PREDICTED_SALES",'sum'))

,MONTH_NAME,TOTAL_MONTHLY_SALES
0,August,311148.39
1,September,260361.83


In [47]:
df_final_output.to_csv(r"Final_output_validation_set.csv",index=False)

### Explainability

In [48]:
from darts.explainability import TFTExplainer

In [49]:
N = 100

explainer = TFTExplainer(
    loaded_model,
    background_series=final_scaled_lookback_data[:N],
    background_future_covariates=final_scaled_lookahead_data[:N],
)
result = explainer.explain()

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting DataLoader 0: 100%|██████████| 1/1 [00:00<00:00,  3.29it/s]


In [50]:
# CHECKPOINT 1 — look at this output, find the right accessor names
print([m for m in dir(result) if not m.startswith('_')])

['available_components', 'explained_components', 'feature_importances', 'get_attention', 'get_decoder_importance', 'get_encoder_importance', 'get_explanation', 'get_feature_importances', 'get_static_covariates_importance']


In [51]:
dec_imp = result.get_decoder_importance()   # <-- adjust if needed
enc_imp = result.get_encoder_importance()   # <-- adjust if needed

In [52]:
print(type(dec_imp))
if isinstance(dec_imp, list):
    import pandas as pd
    dec_imp = pd.concat(dec_imp, axis=0)
    enc_imp = pd.concat(enc_imp, axis=0)

<class 'list'>


In [53]:
dec_top = dec_imp.mean(axis=0).sort_values(ascending=False)
enc_top = enc_imp.mean(axis=0).sort_values(ascending=False)

In [54]:
dec_imp.to_csv('decoder_importance.csv')
enc_imp.to_csv('encoder_importance.csv')

In [59]:
future_covariates

['DUSSEHRA_(VIJAYADASHAMI)_DAYS',
 'NUM_FESTIVE_DAYS_MONTH',
 'AKSHAYA_TRITIYA_DAYS',
 'BHAI_DOOJ_DAYS',
 'BUDDHA_PURNIMA_DAYS',
 'CHHATH_PUJA_DAYS',
 'DHANTERAS_DAYS',
 'DIWALI_DAYS',
 'EID_UL_FITR_DAYS',
 'GANESH_CHATURTHI_DAYS',
 'GANGA_DUSSEHRA_DAYS',
 'GOVARDHAN_POOJA_DAYS',
 'GURU_PURNIMA_DAYS',
 'HANUMAN_JAYANTI_DAYS',
 'HARTALIK_TEEJ_DAYS',
 'HOLI_DAYS',
 'HOLIKA_DAHAN_DAYS',
 'JAGANNATH_RATHYATRA_DAYS',
 'JANMASHTAMI_DAYS',
 'KARWA_CHAUTH_DAYS',
 'LOHRI_DAYS',
 'MAHA_SHIVARATRI_DAYS',
 'MAKAR_SANKRANTI_PONGAL_DAYS',
 'NAG_PANCHAMI_DAYS',
 'NAVRATRI_DAYS',
 'NEW_YEAR_DAYS',
 'ONAM_DAYS',
 'PITRAPAKSHA_DAYS',
 'RAKSHA_BANDHAN_DAYS',
 'REPUBLIC_DAY_DAYS',
 'VASANT_PANCHAMI_DAYS',
 'VISHWAKARMA_PUJA_DAYS',
 'MARRIAGE_DAYS',
 'FESTIVE_PHASE_I',
 'FESTIVE_PHASE_II',
 'FESTIVE_PHASE_III',
 'PITRU_PAKSH',
 'TOTAL_DAYS_FESTIVE_PHASE_I',
 'TOTAL_DAYS_FESTIVE_PHASE_II',
 'TOTAL_DAYS_FESTIVE_PHASE_III',
 'TOTAL_DAYS_PITRU_PAKSH',
 'PROP_FESTIVE_PHASE_I',
 'PROP_EVENT_FESTIVE_PHASE_I',
 'P

In [60]:
final_future_covariates = [i for i in future_covariates if i not in ['NON_ZERO_FLAG','PARENT_DEALER_CODE_MODEL_FAMILY']]
final_future_covariates

['DUSSEHRA_(VIJAYADASHAMI)_DAYS',
 'NUM_FESTIVE_DAYS_MONTH',
 'AKSHAYA_TRITIYA_DAYS',
 'BHAI_DOOJ_DAYS',
 'BUDDHA_PURNIMA_DAYS',
 'CHHATH_PUJA_DAYS',
 'DHANTERAS_DAYS',
 'DIWALI_DAYS',
 'EID_UL_FITR_DAYS',
 'GANESH_CHATURTHI_DAYS',
 'GANGA_DUSSEHRA_DAYS',
 'GOVARDHAN_POOJA_DAYS',
 'GURU_PURNIMA_DAYS',
 'HANUMAN_JAYANTI_DAYS',
 'HARTALIK_TEEJ_DAYS',
 'HOLI_DAYS',
 'HOLIKA_DAHAN_DAYS',
 'JAGANNATH_RATHYATRA_DAYS',
 'JANMASHTAMI_DAYS',
 'KARWA_CHAUTH_DAYS',
 'LOHRI_DAYS',
 'MAHA_SHIVARATRI_DAYS',
 'MAKAR_SANKRANTI_PONGAL_DAYS',
 'NAG_PANCHAMI_DAYS',
 'NAVRATRI_DAYS',
 'NEW_YEAR_DAYS',
 'ONAM_DAYS',
 'PITRAPAKSHA_DAYS',
 'RAKSHA_BANDHAN_DAYS',
 'REPUBLIC_DAY_DAYS',
 'VASANT_PANCHAMI_DAYS',
 'VISHWAKARMA_PUJA_DAYS',
 'MARRIAGE_DAYS',
 'FESTIVE_PHASE_I',
 'FESTIVE_PHASE_II',
 'FESTIVE_PHASE_III',
 'PITRU_PAKSH',
 'TOTAL_DAYS_FESTIVE_PHASE_I',
 'TOTAL_DAYS_FESTIVE_PHASE_II',
 'TOTAL_DAYS_FESTIVE_PHASE_III',
 'TOTAL_DAYS_PITRU_PAKSH',
 'PROP_FESTIVE_PHASE_I',
 'PROP_EVENT_FESTIVE_PHASE_I',
 'P

In [62]:
final_static_covariates = static_covariates + ['PARENT_DEALER_CODE']

In [ ]:
with open(r"C:\Users\G0004878\Desktop\TFT_Data\Temporal-Fusion-Transformers-Material\Documentation for Gourav\static_covariates.txt",'w') as f:
    for feature in final_static_covariates:
        f.write(f"{feature}\n")

TypeError: write() argument must be str, not list